- 일반 Attention vs Multi-Head Attention
(1) 같은 문장에서도 “관계”는 여러 종류라서
예: “나는 어제 은행에 갔다”
“은행”이 finance인지 river bank인지 문맥으로 판단해야 함
어떤 헤드는 “시간/장소 단서”에
다른 헤드는 “주변 단어 의미”에
또 다른 헤드는 “문장 전역 정보”에 집중하는 식으로 동시에 여러 관계를 잡아냄

(2) 긴 문장/복잡한 문맥에서 더 잘 버팀
싱글 attention은 전역을 다 보긴 하지만 “한 가지 정렬”로만 보니까,
복잡한 의존성이 많아질수록 한 번에 잡기 힘든데
MHA는 여러 헤드가 분산해서 잡아주니 안정적.

(3) 병렬 연산이 잘 맞아서(Transformer의 장점 극대화)
RNN처럼 순차가 아니라 행렬곱 중심이라 GPU에서 효율이 좋고,
MHA는 “여러 attention을 병렬로” 돌려도 구조적으로 잘 맞음.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

X = torch.tensor(
    [[1.0, 0.0, 1.0, 2.0],
     [0.0, 2.0, 0.0, 2.0],
     [1.0, 1.0, 1.0, 1.0]]
)
print(X.shape)

torch.Size([3, 4])


In [6]:
W_q = nn.Linear(4, 4, bias=False)
W_k = nn.Linear(4, 4, bias=False)
W_v = nn.Linear(4, 4, bias=False)

Q = W_q(X)
K = W_k(X)
V = W_v(X)

print(Q.shape)
print(K.shape)
print(V.shape)

attn_scores = torch.matmul(Q, K.transpose(-2, -1))
attn_scores /= Q.size(-1) ** 0.5
print(f'attn_scores: {attn_scores.shape}')

attn_weights = F.softmax(attn_scores, dim=1)
print(f'attn_weights: {attn_weights}')

output = torch.matmul(attn_weights, V)
print(f'attn_value: {output.shape}')

torch.Size([3, 4])
torch.Size([3, 4])
torch.Size([3, 4])
attn_scores: torch.Size([3, 3])
attn_weights: tensor([[0.3523, 0.1926, 0.4552],
        [0.2821, 0.2896, 0.4283],
        [0.3022, 0.2869, 0.4108]], grad_fn=<SoftmaxBackward0>)
attn_value: torch.Size([3, 4])


In [10]:

X = torch.tensor(
[    [[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
     [0.0, 2.0, 0.0, 2.0, 0.0, 2.0, 0.0, 2.0],
     [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]]
)
print(X.shape)

B, T, _ = X.shape
embedding_dim = 8
num_head = 4
heading_dim = embedding_dim // num_head

torch.Size([1, 3, 8])


In [13]:
W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

Q = W_q(X)
K = W_k(X)
V = W_v(X)

print(Q.shape)
print(K.shape)
print(V.shape)

Q_head = Q.view(B, T, num_head, heading_dim).transpose(1, 2)
K_head = K.view(B, T, num_head, heading_dim).transpose(1, 2)
V_head = V.view(B, T, num_head, heading_dim).transpose(1, 2)

print(Q_head.shape)
print(K_head.shape)
print(V_head.shape)

attn_scores = torch.matmul(Q_head, K_head.transpose(-2, -1))
attn_scores /= Q.size(-1) ** 0.5
print(f'attn_scores: {attn_scores.shape}')

attn_weights = F.softmax(attn_scores, dim=1)
print(f'attn_weights: {attn_weights.shape}')

output = torch.matmul(attn_weights, V_head)
print(f'attn_value: {output.shape}')

output = output.transpose(1, 2)
output = output.contiguous().view(B, T, embedding_dim)
print(f'헤드 결합 후 출력: {output.shape}')

torch.Size([1, 3, 8])
torch.Size([1, 3, 8])
torch.Size([1, 3, 8])
torch.Size([1, 4, 3, 2])
torch.Size([1, 4, 3, 2])
torch.Size([1, 4, 3, 2])
attn_scores: torch.Size([1, 4, 3, 3])
attn_weights: torch.Size([1, 4, 3, 3])
attn_value: torch.Size([1, 4, 3, 2])
헤드 결합 후 출력: torch.Size([1, 3, 8])
